In [1]:
import pandas as pd
import fnmatch
import numpy as np
from glob import glob

In [112]:
EUT046_normalized = pd.read_csv('../datasets/EUT046_RODAFpreprocessed_normalizedcounts_010824.csv', index_col='Unnamed: 0')

In [113]:
EUT046_normalized.replace(0, 1, inplace=True)

In [114]:
EUT046_normalized_log2 = np.log2(EUT046_normalized)

In [115]:
timepoints = set('_' + a.split('_')[1] + '_' for a in EUT046_normalized.columns)

In [116]:
EUT046_normalized_log2['entrez_id'] = EUT046_normalized_log2.index
EUT046_normalized_log2.entrez_id = EUT046_normalized_log2.entrez_id.apply(lambda x:str(x))

In [117]:
EUT046_normalized_splitbytime = {t:EUT046_normalized_log2.filter(regex=f'{t}|entrez_id') for t in timepoints}

In [118]:
EUT046_normalized_splitbytime

{'_48h_':            S_48h_Cis9_Rep1  S_48h_Cis9_Rep2  S_48h_Cis9_Rep3  S_48h_Cis8_Rep1  \
 2                 1.480746         2.003050         0.000000         0.272016   
 9                 5.235633         5.295832         5.649546         4.972455   
 10                0.000000         2.003050         2.364144         0.000000   
 12                0.000000         0.000000         0.000000         0.000000   
 14                5.650671         5.909941         6.119031         5.629568   
 ...                    ...              ...              ...              ...   
 103091866         1.480746         2.266085         1.364144         4.079371   
 107985423         2.480746         1.266085         0.364144         0.000000   
 107987545         3.288101         0.000000         0.000000         2.593944   
 120356739         3.940178         4.588013         3.171499         3.441941   
 124903857         5.568209         2.681122         0.364144         5.731447   
 
     

In [119]:
EUT046_normalized_splitbytime_cmean = dict()
for k in EUT046_normalized_splitbytime:
    controls = EUT046_normalized_splitbytime[k].filter(like='DMEM')
    controls['mean'] = controls.mean(axis=1)
    controls['entrez_id'] = controls.index
    controls.entrez_id = controls.entrez_id.apply(lambda x:str(x))
    print(controls)
    EUT046_normalized_splitbytime_cmean[k] = pd.merge(EUT046_normalized_splitbytime[k], controls[['entrez_id','mean']], how='left', on='entrez_id')
    
    
    

C:\Users\EmDe5048\AppData\Local\Temp\ipykernel_19892\3177134976.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  controls['mean'] = controls.mean(axis=1)
C:\Users\EmDe5048\AppData\Local\Temp\ipykernel_19892\3177134976.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  controls['entrez_id'] = controls.index
C:\Users\EmDe5048\AppData\Local\Temp\ipykernel_19892\3177134976.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer

           S_48h_DMEM_Rep1  S_48h_DMEM_Rep2  S_48h_DMEM_Rep3  S_48h_DMEM_Rep4  \
2                 2.030919         1.066443         1.264307         2.038456   
9                 5.490351         5.492708         5.656624         5.690532   
10               -0.291009         1.066443         2.001273         0.816063   
12                0.000000         1.651406        -0.320656         1.553029   
14                5.591634         5.949086         6.054384         5.339625   
...                    ...              ...              ...              ...   
103091866        -0.291009         0.000000        -0.320656         1.231101   
107985423         1.708991         2.388371         1.679344         0.816063   
107987545        -0.291009         0.000000        -0.320656         2.231101   
120356739         4.994393         4.590005         5.379784         4.903526   
124903857         3.878916         3.525875         3.586235         4.722954   

           S_48h_DMEM_Rep5 

C:\Users\EmDe5048\AppData\Local\Temp\ipykernel_19892\3177134976.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  controls['mean'] = controls.mean(axis=1)
C:\Users\EmDe5048\AppData\Local\Temp\ipykernel_19892\3177134976.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  controls['entrez_id'] = controls.index
C:\Users\EmDe5048\AppData\Local\Temp\ipykernel_19892\3177134976.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer

In [120]:
EUT046_fgsea = dict()
for t in EUT046_normalized_splitbytime_cmean.keys():
    df = EUT046_normalized_splitbytime_cmean[t]
    cols_to_subtract = df.filter(like='Cis').columns.to_list()
    df[cols_to_subtract] = df[cols_to_subtract].subtract(df['mean'], axis=0)
    EUT046_fgsea[t] = df

In [145]:
manifest = pd.read_csv('../datasets/Temposeq_manifest_Human_WT_1.2_realignment_96percent_2023-05-15 (1).txt', sep='\t')

In [185]:
for t,df in EUT046_fgsea.items():
    df['entrez_id'] = df.entrez_id.apply(lambda x:int(x))
    #df = pd.merge(df, manifest[['gene_symbol', 'entrez_id']], how='left', on='entrez_id')
    cols = ['entrez_id']
    colsbis = df.filter(like='Cis').columns
    for col in colsbis:
        cols.append(col)
    final_df = df[cols]
    final_df.drop_duplicates(inplace=True)
    filename= f'EUT046_{t}_fgsea.csv'
    final_df.to_csv(f'../fgsea_files/input/{filename}', sep='\t', index=False)

C:\Users\EmDe5048\AppData\Local\Temp\ipykernel_19892\2427075609.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.drop_duplicates(inplace=True)
C:\Users\EmDe5048\AppData\Local\Temp\ipykernel_19892\2427075609.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.drop_duplicates(inplace=True)
C:\Users\EmDe5048\AppData\Local\Temp\ipykernel_19892\2427075609.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.drop

In [22]:
fgsea_resfiles = glob('../fgsea_files/output/*')

In [23]:
fgsea_res_d = {file.split('output\\')[-1].split('.csv')[0]:pd.read_csv(file, sep='\t') for file in fgsea_resfiles}

In [24]:
fgsea_resfiles

['../fgsea_files/output\\S_16h_Cis1_Rep1.csv',
 '../fgsea_files/output\\S_16h_Cis1_Rep2.csv',
 '../fgsea_files/output\\S_16h_Cis1_Rep3.csv',
 '../fgsea_files/output\\S_16h_Cis2_Rep1.csv',
 '../fgsea_files/output\\S_16h_Cis2_Rep2.csv',
 '../fgsea_files/output\\S_16h_Cis2_Rep3.csv',
 '../fgsea_files/output\\S_16h_Cis3_Rep1.csv',
 '../fgsea_files/output\\S_16h_Cis3_Rep2.csv',
 '../fgsea_files/output\\S_16h_Cis3_Rep3.csv',
 '../fgsea_files/output\\S_16h_Cis4_Rep1.csv',
 '../fgsea_files/output\\S_16h_Cis4_Rep2.csv',
 '../fgsea_files/output\\S_16h_Cis4_Rep3.csv',
 '../fgsea_files/output\\S_16h_Cis5_Rep1.csv',
 '../fgsea_files/output\\S_16h_Cis5_Rep2.csv',
 '../fgsea_files/output\\S_16h_Cis5_Rep3.csv',
 '../fgsea_files/output\\S_16h_Cis6_Rep1.csv',
 '../fgsea_files/output\\S_16h_Cis6_Rep2.csv',
 '../fgsea_files/output\\S_16h_Cis6_Rep3.csv',
 '../fgsea_files/output\\S_16h_Cis7_Rep1.csv',
 '../fgsea_files/output\\S_16h_Cis7_Rep2.csv',
 '../fgsea_files/output\\S_16h_Cis7_Rep3.csv',
 '../fgsea_fi

In [25]:
fgsea_res_forBMD = pd.concat([fgsea_res_d['S_16h_Cis1_Rep1'][['pathway']], pd.DataFrame({k:v.NES for k,v in fgsea_res_d.items()})], axis=1)

In [7]:
fgsea_res_forBMD.rename(columns={'pathway':'0_0_Dose'}, inplace=True)

In [10]:
fgsea_res_forBMD = pd.concat([fgsea_res_forBMD, pd.DataFrame({col:col.split('_')[2] for col in fgsea_res_forBMD.columns}, index=[1])])


In [27]:
fgsea_res_forBMD_long = pd.melt(fgsea_res_forBMD, id_vars=['pathway'], value_vars=fgsea_res_forBMD.columns.to_list()[1:], value_name='fGSEA_NES')

In [31]:
fgsea_res_forBMD_long.rename(columns={'variable':'sample_id'}, inplace=True)

In [32]:
fgsea_res_forBMD_long

,pathway,sample_id,fGSEA_NES
0,GENOMARK84,S_16h_Cis1_Rep1,1.642692
1,HALLMARK_ADIPOGENESIS,S_16h_Cis1_Rep1,-0.965309
2,HALLMARK_ALLOGRAFT_REJECTION,S_16h_Cis1_Rep1,0.907270
3,HALLMARK_ANDROGEN_RESPONSE,S_16h_Cis1_Rep1,-1.340396
4,HALLMARK_ANGIOGENESIS,S_16h_Cis1_Rep1,-1.128153
...,...,...,...
8419,HALLMARK_UV_RESPONSE_DN,S_8h_Cis9_Rep3,0.667131
8420,HALLMARK_UV_RESPONSE_UP,S_8h_Cis9_Rep3,0.895794
8421,HALLMARK_WNT_BETA_CATENIN_SIGNALING,S_8h_Cis9_Rep3,1.153106
8422,HALLMARK_XENOBIOTIC_METABOLISM,S_8h_Cis9_Rep3,0.906560


In [33]:
fgsea_res_forBMD_long.to_csv('../data/input/EUT046_sciensano/EUT046_NES.txt', sep='\t', index=False)

In [21]:
fgsea_res_forBMD.to_csv('../data/input/EUT046_sciensano/EUT046_NES.txt', sep='\t', index=False)

In [48]:
conc = {'DMEM':0, 'Cis9':0.01, 'Cis8':0.5, 'Cis7':1, 'Cis6':2.5, 'Cis5':5, 'Cis4':10, 'Cis3':20, 'Cis2':30, 'Cis1':50}

In [49]:
fgsea_res_forBMD.replace(conc, inplace=True)

In [51]:
derniere_ligne = fgsea_res_forBMD.iloc[-1]
fgsea_res_forBMD = pd.concat([derniere_ligne.to_frame().T, fgsea_res_forBMD.iloc[:-1]])

In [52]:
fgsea_res_forBMD.reset_index(drop=True, inplace=True)

In [58]:
fgsea_res_forBMD.fillna(0, inplace=True)

In [60]:
timepoint = ['_4h', '_8h', '_16h', '_24h', '_48h', '_72h']

In [63]:
fgsea_res_forBMD_timepoint = {t:fgsea_res_forBMD.filter(regex=f'Dose|{t}') for t in timepoint}

In [68]:
for t,v in fgsea_res_forBMD_timepoint.items():
    print(v.shape)
    v.to_csv(f'../datasets/EUT046_NES_forBMD/EUT046_NES{t}.txt', sep = '\t', index=False)

(53, 28)
(53, 28)
(53, 28)
(53, 28)
(53, 28)
(53, 28)
